In [ ]:
import os
import json
import time
import requests
import pandas as pd
from dotenv import load_dotenv
from tqdm import tqdm
import psycopg2
from psycopg2.extras import Json

---

#### Authentification INPI

In [ ]:
# # 1. Chargement des variables d'environnement
# load_dotenv()

# USERNAME = os.getenv("INPI_USERNAME")
# PASSWORD = os.getenv("INPI_PASSWORD")

# if not USERNAME or not PASSWORD:
#     raise RuntimeError("❌ INPI_USERNAME ou INPI_PASSWORD manquant dans le .env")

# # 2. Authentification et récupération du Token
# LOGIN_URL = "https://registre-national-entreprises.inpi.fr/api/sso/login"

# payload = {
#     "username": USERNAME,
#     "password": PASSWORD,
# }

# try:
#     print("🔑 Tentative de connexion à l'INPI...")
#     response = requests.post(LOGIN_URL, json=payload, timeout=15)
#     response.raise_for_status()
    
#     token = response.json().get("token")
    
#     if not token:
#         raise ValueError("Le format de réponse de l'API ne contient pas de token.")
        
#     print(f"✅ Authentification réussie !")
#     print(f"🎫 Token OK : {token[:20]}...")

#     # 3. Configuration des Headers pour la suite
#     HEADERS = {
#         "Authorization": f"Bearer {token}",
#         "Content-Type": "application/json"
#     }

# except requests.exceptions.HTTPError as err:
#     print(f"❌ Erreur HTTP lors de la connexion : {err}")
# except Exception as e:
#     print(f"❌ Erreur inattendue : {e}")

---

### Acquisition des bilans

#### Import des SIREN

In [ ]:
# with open("siren_sample.json", "r", encoding="utf-8") as f:
#     sirens_a_traiter = json.load(f)

# print(f"📂 {len(sirens_a_traiter)} SIREN chargés depuis le fichier.")

#### Connection SQL

In [ ]:
# load_dotenv()

# DATABASE_URL = os.getenv("NEON_DATABASE_URL")

# if not DATABASE_URL:
#     raise ValueError("La variable NEON_DATABASE_URL n'est pas définie dans .env")

# try:
#     # On ajoute sslmode=require pour Neon
#     conn = psycopg2.connect(DATABASE_URL)
#     cur = conn.cursor()
    
#     # Test pour confirmer la version de la BDD
#     cur.execute("SELECT version();")
#     db_version = cur.fetchone()
#     print(f"✅ Connexion à Néon réussie !")
#     print(f"🖥️ Version : {db_version[0][:30]}...")

# except Exception as e:
#     print(f"❌ Erreur de connexion : {e}")

#### Acquisition des bilans des 500 premiers SIREN

In [ ]:
# def extraire_bilans_dictionnaire(data):
#     """Transforme le JSON complexe de l'INPI en dictionnaire 'Key-Value'"""
#     bilans = []
#     if "bilansSaisis" in data:
#         for b in data["bilansSaisis"]:
#             liasses_dict = {}
#             try:
#                 # On descend dans la structure spécifique de l'API RNE
#                 pages = b.get('bilanSaisi', {}).get('bilan', {}).get('detail', {}).get('pages', [])
#                 for page in pages:
#                     for liasse in page.get('liasses', []):
#                         code = liasse.get("code")
#                         # On récupère m1 (valeur brute), par défaut 0
#                         valeur = int(liasse.get("m1") or 0)
#                         if code:
#                             liasses_dict[code] = valeur
                
#                 bilans.append({
#                     "siren": b.get("siren"),
#                     "date_cloture": b.get("dateCloture"),
#                     "confidentialite": b.get("confidentiality"),
#                     "liasses": liasses_dict
#                 })
#             except (KeyError, TypeError) as e:
#                 # Si un bilan est mal formé, on passe au suivant
#                 continue
#     return bilans

# # --- CONFIGURATION ---
# BACKUP_DIR = "backup_bilans"
# os.makedirs(BACKUP_DIR, exist_ok=True)

# # 1. SYNCHRONISATION : On vérifie ce qu'on a déjà (Local + Neon)

# local_files = {f.replace(".json", "") for f in os.listdir(BACKUP_DIR)}

# # Vérification BDD Neon
# cur.execute("SELECT DISTINCT siren FROM bilans_entreprises")
# neon_sirens = {row[0] for row in cur.fetchall()}

# # On ne traite que ce qui n'est NI en local, NI sur Neon
# deja_faits = local_files.union(neon_sirens)
# sirens_restants = [s for s in sirens_a_traiter if s not in deja_faits]

# print(f"📊 État des lieux :")
# print(f"📁 Archive locale : {len(local_files)} entreprises")
# print(f"☁️  Base Neon      : {len(neon_sirens)} entreprises")
# print(f"🚀 Reste à traiter : {len(sirens_restants)}")
# print("-" * 50)

# # 2. BOUCLE D'ACQUISITION
# for i, siren in enumerate(sirens_restants):
#     print(f"⏳ [{i+1}/{len(sirens_restants)}] Traitement : {siren}...", end="\r")
    
#     try:
#         url = f"https://registre-national-entreprises.inpi.fr/api/companies/{siren}/attachments"
#         res = requests.get(url, headers=HEADERS, timeout=20)
        
#         if res.status_code == 200:
#             data = res.json()
#             bilans_propres = extraire_bilans_dictionnaire(data)
            
#             # SAUVEGARDE LOCALE (Fichier individuel)
#             with open(f"{BACKUP_DIR}/{siren}.json", "w", encoding="utf-8") as f:
#                 json.dump(bilans_propres, f, indent=2, ensure_ascii=False)
            
#             # INJECTION NEON
#             if bilans_propres:
#                 insert_query = """
#                 INSERT INTO bilans_entreprises (siren, date_cloture, confidentialite, donnees_liasses)
#                 VALUES (%s, %s, %s, %s)
#                 ON CONFLICT (siren, date_cloture) DO NOTHING;
#                 """
#                 for b in bilans_propres:
#                     cur.execute(insert_query, (
#                         b['siren'], 
#                         b['date_cloture'], 
#                         b['confidentialite'], 
#                         Json(b['liasses'])
#                     ))
#                 conn.commit()
                
#         elif res.status_code == 429:
#             print(f"\n⚠️ Rate limit ! Pause de 60s...")
#             time.sleep(60)
#             # Optionnel: interrompre ici ou continuer
            
#     except Exception as e:
#         print(f"\n❌ Erreur sur {siren} : {e}")
#         conn.rollback()

#     # Respect du quota INPI
#     time.sleep(5.5)

# print("\n\n✅ Mission terminée. Ton dossier 'backup_bilans' et ta base Neon sont synchronisés.")

---

#### Augmentation des bilans à acquérir avec l'échantillon à 2000 SIREN

In [ ]:
# # --- 1. FONCTIONS DE BASE ---

# def extraire_bilans_dictionnaire(data):
#     """
#     LOGIQUE AMÉLIORÉE : Récupère TOUTE l'antériorité (N, N-1, N-2, N-3).
#     Si une société est fermée, cela permet d'avoir les 3 bilans avant la clôture.
#     """
#     res_list = []
#     if not data: return res_list

#     saisis = data.get('bilansSaisis', [])
#     pdfs = data.get('bilans', [])
#     actes = data.get('actes', [])

#     # Dictionnaire pour dédoublonner par date de clôture 
#     # (on préfère une donnée STRUCT à un PDF si les deux existent pour la même date)
#     bilans_par_date = {}

#     mots_cles = ["bilan", "comptes annuels", "liasse", "comptes de l'exercice", "clôture"]

#     # ÉTAPE 1 : On aspire les données structurées (le Graal pour XGBoost)
#     for b in saisis:
#         date_c = b.get('dateCloture')
#         if date_c:
#             bilans_par_date[str(date_c)] = {
#                 'siren': b.get('siren'),
#                 'date_cloture': date_c,
#                 'confidentialite': b.get('confidentiality', 'Public'),
#                 'liasses': b,
#                 'source': 'STRUCT'
#             }

#     # ÉTAPE 2 : On complète avec les PDF (bilans et actes) pour boucher les trous de l'historique
#     for doc in pdfs + actes:
#         libelle = doc.get('libelle', '').lower()
        
#         # Analyse RDD pour ne rien rater dans les dossiers complexes
#         rdd_decisions = ""
#         if 'typeRdd' in doc:
#             rdd_decisions = " ".join([str(r.get('decision', '')).lower() for r in doc['typeRdd']])
        
#         date_c = doc.get('dateCloture') or doc.get('dateDepot')
        
#         # Si c'est un document comptable et qu'on n'a pas encore de donnée pour cette date
#         if any(mot in libelle for mot in mots_cles) or any(mot in rdd_decisions for mot in mots_cles):
#             if date_c and str(date_c) not in bilans_par_date:
#                 bilans_par_date[str(date_c)] = {
#                     'siren': doc.get('siren'),
#                     'date_cloture': date_c,
#                     'confidentialite': doc.get('confidentiality', 'Public'),
#                     'liasses': doc,
#                     'source': 'PDF_DOCUMENT'
#                 }

#     return list(bilans_par_date.values())

# def refresh_inpi_token():
#     LOGIN_URL = "https://registre-national-entreprises.inpi.fr/api/sso/login"
#     payload = {"username": os.getenv("INPI_USERNAME"), "password": os.getenv("INPI_PASSWORD")}
#     try:
#         print("\n🔑 Tentative de connexion à l'INPI...")
#         response = requests.post(LOGIN_URL, json=payload, timeout=15)
#         response.raise_for_status()
#         token = response.json().get("token")
#         print(f"✅ Authentification réussie !")
#         return token
#     except Exception as e:
#         print(f"❌ Erreur d'authentification : {e}")
#         return None

# # --- 2. INITIALISATION ---
# load_dotenv()
# BACKUP_DIR = "backup_bilans"
# INPUT_FILE = "siren_batch_2000.json"
# if not os.path.exists(BACKUP_DIR): os.makedirs(BACKUP_DIR)

# DATABASE_URL = os.getenv("NEON_DATABASE_URL")
# try:
#     conn = psycopg2.connect(DATABASE_URL)
#     cur = conn.cursor()
#     print("✅ Connexion à Néon réussie !")
# except Exception as e:
#     print(f"❌ Erreur de connexion BDD : {e}"); exit()

# token = refresh_inpi_token()
# if not token: exit()
# HEADERS = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

# # --- 3. NETTOYAGE ---
# print("🧹 Nettoyage des résidus de scans vides...")
# files_removed = 0
# for f in os.listdir(BACKUP_DIR):
#     if f.endswith(".json"):
#         path = os.path.join(BACKUP_DIR, f)
#         if os.path.getsize(path) < 10:
#             os.remove(path)
#             files_removed += 1
# print(f"🗑️  {files_removed} fichiers supprimés.")

# # --- 4. FILTRAGE ---
# with open(INPUT_FILE, 'r') as f:
#     tous_sirens = json.load(f)

# local_files = {f.replace(".json", "") for f in os.listdir(BACKUP_DIR) if f.endswith(".json")}
# sirens_restants = [s for s in tous_sirens if s not in local_files]

# print(f"\n📊 ANALYSE DU BATCH : {len(sirens_restants)} à traiter.")
# print("-" * 50)

# # --- 5. BOUCLE D'ACQUISITION ---
# for i, siren in enumerate(sirens_restants):
#     print(f"⏳ [{i+1}/{len(sirens_restants)}] SIREN: {siren}", end="\r")
#     url = f"https://registre-national-entreprises.inpi.fr/api/companies/{siren}/attachments"
    
#     try:
#         # VERIFICATION DE LA CONNEXION BDD (Anti-SSL Error)
#         if conn.closed != 0:
#             print("\n🔄 Reconnexion à Néon...")
#             conn = psycopg2.connect(DATABASE_URL)
#             cur = conn.cursor()

#         res = requests.get(url, headers=HEADERS, timeout=20)
        
#         if res.status_code == 401:
#             token = refresh_inpi_token()
#             if token:
#                 HEADERS["Authorization"] = f"Bearer {token}"
#                 res = requests.get(url, headers=HEADERS, timeout=20)
#             else: time.sleep(60); continue

#         if res.status_code == 200:
#             data = res.json()
#             if any(k in data for k in ['actes', 'bilans', 'bilansSaisis']):
#                 bilans = extraire_bilans_dictionnaire(data)
                
#                 # Sauvegarde locale (marquage comme traité)
#                 with open(f"{BACKUP_DIR}/{siren}.json", "w", encoding="utf-8") as f:
#                     json.dump(bilans, f, indent=2, ensure_ascii=False)
                
#                 if bilans:
#                     print(f"\n💰 {len(bilans)} exercices pour {siren}. Insertion...")
#                     try:
#                         for b in bilans:
#                             cur.execute("""
#                                 INSERT INTO bilans_entreprises (siren, date_cloture, confidentialite, donnees_liasses)
#                                 VALUES (%s, %s, %s, %s)
#                                 ON CONFLICT (siren, date_cloture) DO NOTHING;
#                             """, (b['siren'], b['date_cloture'], b['confidentialite'], Json(b['liasses'])))
#                         conn.commit()
#                     except (psycopg2.InterfaceError, psycopg2.OperationalError):
#                         print("⚠️ Perte de connexion pendant l'insertion, reconnexion...")
#                         conn = psycopg2.connect(DATABASE_URL)
#                         cur = conn.cursor()
#                         # On ne refait pas l'insert ici, le prochain SIREN le fera ou le fichier JSON local évitera de boucler
#             else:
#                 with open(f"{BACKUP_DIR}/{siren}.json", "w") as f: json.dump([], f)

#         elif res.status_code == 429:
#             print(f"\n⚠️ Rate limit. Pause 60s...")
#             time.sleep(60)
#         elif res.status_code == 404:
#             with open(f"{BACKUP_DIR}/{siren}.json", "w") as f: json.dump([], f)

#     except Exception as e:
#         print(f"\n❌ Erreur sur {siren}: {e}")
#         # On ne fait pas rollback() si la connexion est fermée
#         if conn and conn.closed == 0:
#             conn.rollback()

#     time.sleep(5.1)

# print("\n\n✨ TERMINÉ ! Historique complet aspiré.")
# cur.close(); conn.close()

---

#### Passage à 20 000 SIREN à récupérer

In [ ]:
# import os
# import json
# import time
# import requests
# import psycopg2
# from psycopg2.extras import Json
# from dotenv import load_dotenv

# # --- 1. FONCTIONS DE BASE ---

# def extraire_bilans_dictionnaire(data):
#     res_list = []
#     if not data: return res_list

#     saisis = data.get('bilansSaisis', [])
#     pdfs = data.get('bilans', [])
#     actes = data.get('actes', [])

#     bilans_par_date = {}
#     mots_cles = ["bilan", "comptes annuels", "liasse", "comptes de l'exercice", "clôture"]

#     # ÉTAPE 1 : Données structurées
#     for b in saisis:
#         date_c = b.get('dateCloture')
#         if date_c:
#             # ON RÉCUPÈRE LE CONTENU NICHÉ 'bilanSaisi' s'il existe, sinon l'objet b
#             contenu_liasse = b.get('bilanSaisi', b)
            
#             bilans_par_date[str(date_c)] = {
#                 'siren': b.get('siren'),
#                 'date_cloture': date_c,
#                 'confidentialite': b.get('confidentiality', 'Public'),
#                 'liasses': contenu_liasse, # <-- C'est ici que ça se jouait
#                 'source': 'STRUCT'
#             }

#     # ÉTAPE 2 : PDF et Actes (inchangée)
#     for doc in pdfs + actes:
#         libelle = str(doc.get('libelle', '')).lower()
#         rdd_decisions = ""
#         if 'typeRdd' in doc:
#             rdd_decisions = " ".join([str(r.get('decision', '')).lower() for r in doc['typeRdd']])
#         date_c = doc.get('dateCloture') or doc.get('dateDepot')
#         if any(mot in libelle for mot in mots_cles) or any(mot in rdd_decisions for mot in mots_cles):
#             if date_c and str(date_c) not in bilans_par_date:
#                 bilans_par_date[str(date_c)] = {
#                     'siren': doc.get('siren'),
#                     'date_cloture': date_c,
#                     'confidentialite': doc.get('confidentiality', 'Public'),
#                     'liasses': doc,
#                     'source': 'PDF_DOCUMENT'
#                 }
#     return list(bilans_par_date.values())

# def refresh_inpi_token():
#     LOGIN_URL = "https://registre-national-entreprises.inpi.fr/api/sso/login"
#     payload = {"username": os.getenv("INPI_USERNAME"), "password": os.getenv("INPI_PASSWORD")}
#     try:
#         print("\n🔑 Tentative de connexion à l'INPI...")
#         response = requests.post(LOGIN_URL, json=payload, timeout=15)
#         response.raise_for_status()
#         token = response.json().get("token")
#         print(f"✅ Authentification réussie !")
#         return token
#     except Exception as e:
#         print(f"❌ Erreur d'authentification : {e}")
#         return None

# # --- 2. INITIALISATION ---
# load_dotenv()
# BACKUP_DIR = "backup_bilans_batch_2"
# INPUT_FILE = "siren_batch_20000_nouveaux.json"
# if not os.path.exists(BACKUP_DIR): os.makedirs(BACKUP_DIR)

# DATABASE_URL = os.getenv("NEON_DATABASE_URL")
# try:
#     conn = psycopg2.connect(DATABASE_URL)
#     cur = conn.cursor()
#     print("✅ Connexion à Néon réussie !")
# except Exception as e:
#     print(f"❌ Erreur de connexion BDD : {e}"); exit()

# token = refresh_inpi_token()
# if not token: exit()
# HEADERS = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

# # --- 3. NETTOYAGE ---
# print("🧹 Nettoyage des résidus de scans vides...")
# files_removed = 0
# for f in os.listdir(BACKUP_DIR):
#     if f.endswith(".json"):
#         path = os.path.join(BACKUP_DIR, f)
#         if os.path.getsize(path) < 10:
#             os.remove(path)
#             files_removed += 1
# print(f"🗑️  {files_removed} fichiers supprimés.")

# # --- 4. FILTRAGE ---
# with open(INPUT_FILE, 'r') as f:
#     tous_sirens = json.load(f)

# local_files = {f.replace(".json", "") for f in os.listdir(BACKUP_DIR) if f.endswith(".json")}
# sirens_restants = [s for s in tous_sirens if s not in local_files]

# print(f"\n📊 ANALYSE DU BATCH : {len(sirens_restants)} à traiter.")
# print("-" * 50)

# # --- 5. BOUCLE D'ACQUISITION ---
# total_ajouts = 0

# for i, siren in enumerate(sirens_restants):
#     # Affichage simple sur une seule ligne
#     print(f"⏳ [{i+1}/{len(sirens_restants)}] SIREN: {siren} | Ajouts: {total_ajouts}", end="\r")
#     url = f"https://registre-national-entreprises.inpi.fr/api/companies/{siren}/attachments"
    
#     try:
#         if conn.closed != 0:
#             print("\n🔄 Reconnexion à Néon...")
#             conn = psycopg2.connect(DATABASE_URL)
#             cur = conn.cursor()

#         res = requests.get(url, headers=HEADERS, timeout=20)
        
#         if res.status_code == 401:
#             token = refresh_inpi_token()
#             if token:
#                 HEADERS["Authorization"] = f"Bearer {token}"
#                 res = requests.get(url, headers=HEADERS, timeout=20)
#             else: 
#                 time.sleep(60)
#                 continue

#         if res.status_code == 200:
#             data = res.json()
#             if any(k in data for k in ['actes', 'bilans', 'bilansSaisis']):
#                 bilans = extraire_bilans_dictionnaire(data)
                
#                 with open(f"{BACKUP_DIR}/{siren}.json", "w", encoding="utf-8") as f:
#                     json.dump(bilans, f, indent=2, ensure_ascii=False)
                
#                 if bilans:
#                     print(f"\n💰 {len(bilans)} exercices pour {siren}. Insertion...")
#                     try:
#                         for b in bilans:
#                             cur.execute("""
#                                 INSERT INTO bilans_entreprises (siren, date_cloture, confidentialite, donnees_liasses)
#                                 VALUES (%s, %s, %s, %s)
#                                 ON CONFLICT (siren, date_cloture) DO NOTHING;
#                             """, (b['siren'], b['date_cloture'], b['confidentialite'], Json(b['liasses'])))
#                         conn.commit()
#                         total_ajouts += len(bilans)
#                     except (psycopg2.InterfaceError, psycopg2.OperationalError):
#                         print("⚠️ Perte de connexion pendant l'insertion, reconnexion...")
#                         conn = psycopg2.connect(DATABASE_URL)
#                         cur = conn.cursor()
#             else:
#                 with open(f"{BACKUP_DIR}/{siren}.json", "w") as f: json.dump([], f)

#         elif res.status_code == 429:
#             print(f"\n⚠️ Rate limit. Pause 60s...")
#             time.sleep(60)
#         elif res.status_code == 404:
#             with open(f"{BACKUP_DIR}/{siren}.json", "w") as f: json.dump([], f)

#     except Exception as e:
#         print(f"\n❌ Erreur sur {siren}: {e}")
#         if conn and conn.closed == 0:
#             conn.rollback()

#     time.sleep(5.1)

# print(f"\n\n✨ TERMINÉ ! {total_ajouts} nouveaux exercices ajoutés.")
# cur.close(); conn.close()

#### Nouvelle boucle améliorée tenant compte des SIREN déjà balayés.

In [ ]:
# import os
# import json
# import time
# import requests
# import psycopg2
# from psycopg2.extras import Json
# from dotenv import load_dotenv

# # --- 1. FONCTIONS DE BASE ---

# def extraire_bilans_dictionnaire(data):
#     res_list = []
#     if not data: return res_list

#     saisis = data.get('bilansSaisis', [])
#     pdfs = data.get('bilans', [])
#     actes = data.get('actes', [])

#     bilans_par_date = {}
#     mots_cles = ["bilan", "comptes annuels", "liasse", "comptes de l'exercice", "clôture"]

#     # ÉTAPE 1 : Données structurées
#     for b in saisis:
#         date_c = b.get('dateCloture')
#         if date_c:
#             # ON RÉCUPÈRE LE CONTENU NICHÉ 'bilanSaisi' s'il existe, sinon l'objet b
#             contenu_liasse = b.get('bilanSaisi', b)
            
#             bilans_par_date[str(date_c)] = {
#                 'siren': b.get('siren'),
#                 'date_cloture': date_c,
#                 'confidentialite': b.get('confidentiality', 'Public'),
#                 'liasses': contenu_liasse, # <-- C'est ici que ça se jouait
#                 'source': 'STRUCT'
#             }

#     # ÉTAPE 2 : PDF et Actes (inchangée)
#     for doc in pdfs + actes:
#         libelle = str(doc.get('libelle', '')).lower()
#         rdd_decisions = ""
#         if 'typeRdd' in doc:
#             rdd_decisions = " ".join([str(r.get('decision', '')).lower() for r in doc['typeRdd']])
#         date_c = doc.get('dateCloture') or doc.get('dateDepot')
#         if any(mot in libelle for mot in mots_cles) or any(mot in rdd_decisions for mot in mots_cles):
#             if date_c and str(date_c) not in bilans_par_date:
#                 bilans_par_date[str(date_c)] = {
#                     'siren': doc.get('siren'),
#                     'date_cloture': date_c,
#                     'confidentialite': doc.get('confidentiality', 'Public'),
#                     'liasses': doc,
#                     'source': 'PDF_DOCUMENT'
#                 }
#     return list(bilans_par_date.values())

# def refresh_inpi_token():
#     LOGIN_URL = "https://registre-national-entreprises.inpi.fr/api/sso/login"
#     payload = {"username": os.getenv("INPI_USERNAME"), "password": os.getenv("INPI_PASSWORD")}
#     try:
#         print("\n🔑 Tentative de connexion à l'INPI...")
#         response = requests.post(LOGIN_URL, json=payload, timeout=15)
#         response.raise_for_status()
#         token = response.json().get("token")
#         print(f"✅ Authentification réussie !")
#         return token
#     except Exception as e:
#         print(f"❌ Erreur d'authentification : {e}")
#         return None

# # --- 2. INITIALISATION ---
# load_dotenv()
# BACKUP_DIR = "backup_bilans_batch_2"
# INPUT_FILE = "siren_batch_20000_nouveaux.json"
# if not os.path.exists(BACKUP_DIR): os.makedirs(BACKUP_DIR)

# DATABASE_URL = os.getenv("NEON_DATABASE_URL")
# try:
#     conn = psycopg2.connect(DATABASE_URL)
#     cur = conn.cursor()
#     print("✅ Connexion à Néon réussie !")
# except Exception as e:
#     print(f"❌ Erreur de connexion BDD : {e}"); exit()

# token = refresh_inpi_token()
# if not token: exit()
# HEADERS = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

# # --- 3. NETTOYAGE ---
# print("🧹 Nettoyage des résidus de scans vides...")
# files_removed = 0
# for f in os.listdir(BACKUP_DIR):
#     if f.endswith(".json"):
#         path = os.path.join(BACKUP_DIR, f)
#         if os.path.getsize(path) < 10:
#             os.remove(path)
#             files_removed += 1
# print(f"🗑️  {files_removed} fichiers supprimés.")

# # --- 4. FILTRAGE ---
# with open(INPUT_FILE, 'r') as f:
#     tous_sirens = json.load(f)

# local_files = {f.replace(".json", "") for f in os.listdir(BACKUP_DIR) if f.endswith(".json")}
# sirens_restants = [s for s in tous_sirens if s not in local_files]

# print(f"\n📊 ANALYSE DU BATCH : {len(sirens_restants)} à traiter.")
# print("-" * 50)

# # --- 5. BOUCLE D'ACQUISITION OPTIMISÉE ---
# total_ajouts = 0
# idx = 0

# while idx < len(sirens_restants):
#     siren = sirens_restants[idx]
#     print(f"⏳ [{idx+1}/{len(sirens_restants)}] SIREN: {siren} | Ajouts: {total_ajouts}", end="\r")
    
#     url = f"https://registre-national-entreprises.inpi.fr/api/companies/{siren}/attachments"
    
#     try:
#         # Vérification connexion BDD
#         if conn.closed != 0:
#             print("\n🔄 Reconnexion à Néon...")
#             conn = psycopg2.connect(DATABASE_URL)
#             cur = conn.cursor()

#         res = requests.get(url, headers=HEADERS, timeout=20)
        
#         # 1. GESTION AUTHENTIFICATION
#         if res.status_code == 401:
#             token = refresh_inpi_token()
#             if token:
#                 HEADERS["Authorization"] = f"Bearer {token}"
#                 continue # Retente le même SIREN avec le nouveau token
#             else: 
#                 print("\n😴 Échec Refresh Token. Pause 5min...")
#                 time.sleep(300)
#                 continue

#         # 2. GESTION RATE LIMIT (LE POINT CRITIQUE)
#         if res.status_code == 429:
#             print(f"\n🚫 Rate Limit (429). Pause de sécurité 5min pour le SIREN {siren}...")
#             time.sleep(300) # On attend vraiment pour laisser souffler l'API
#             continue # ON NE PASSE PAS AU SUIVANT (idx n'augmente pas)

#         # 3. GESTION SUCCÈS
#         if res.status_code == 200:
#             data = res.json()
#             bilans = []
#             if any(k in data for k in ['actes', 'bilans', 'bilansSaisis']):
#                 bilans = extraire_bilans_dictionnaire(data)
                
#                 if bilans:
#                     print(f"\n💰 {len(bilans)} exercices pour {siren}. Insertion...")
#                     try:
#                         for b in bilans:
#                             cur.execute("""
#                                 INSERT INTO bilans_entreprises (siren, date_cloture, confidentialite, donnees_liasses)
#                                 VALUES (%s, %s, %s, %s)
#                                 ON CONFLICT (siren, date_cloture) DO NOTHING;
#                             """, (b['siren'], b['date_cloture'], b['confidentialite'], Json(b['liasses'])))
#                         conn.commit()
#                         total_ajouts += len(bilans)
#                     except (psycopg2.InterfaceError, psycopg2.OperationalError):
#                         print("⚠️ Perte BDD, reconnexion...")
#                         conn = psycopg2.connect(DATABASE_URL)
#                         cur = conn.cursor()
            
#             # Sauvegarde locale (même si vide) pour marquer comme "fait"
#             with open(f"{BACKUP_DIR}/{siren}.json", "w", encoding="utf-8") as f:
#                 json.dump(bilans, f, indent=2, ensure_ascii=False)
            
#             idx += 1 # Succès : passage au SIREN suivant

#         # 4. GESTION NOT FOUND
#         elif res.status_code == 404:
#             with open(f"{BACKUP_DIR}/{siren}.json", "w") as f: 
#                 json.dump([], f)
#             idx += 1

#         else:
#             print(f"\n⚠️ Code erreur inattendu {res.status_code} sur {siren}. Pause 10s...")
#             time.sleep(10)
#             idx += 1 # On passe pour éviter de boucler à l'infini sur une erreur 500

#     except Exception as e:
#         print(f"\n❌ Erreur sur {siren}: {e}")
#         time.sleep(5)
#         # On ne commit pas, on laisse le SIREN dans la file pour la prochaine itération ou retry

#     # Délai de courtoisie entre chaque SIREN réussi
#     time.sleep(5.1)

# print(f"\n\n✨ TERMINÉ ! {total_ajouts} nouveaux exercices ajoutés.")

---

#### Nouvelle boucle d'acquisition excluant les SIREN déjà traités

In [ ]:
# import os
# import json
# import time
# import requests
# import psycopg2
# from psycopg2.extras import Json
# from dotenv import load_dotenv

# # --- 1. FONCTIONS DE BASE ---

# def extraire_bilans_dictionnaire(data):
#     res_list = []
#     if not data: return res_list

#     saisis = data.get('bilansSaisis', [])
#     pdfs = data.get('bilans', [])
#     actes = data.get('actes', [])

#     bilans_par_date = {}
#     mots_cles = ["bilan", "comptes annuels", "liasse", "comptes de l'exercice", "clôture"]

#     for b in saisis:
#         date_c = b.get('dateCloture')
#         if date_c:
#             contenu_liasse = b.get('bilanSaisi', b)
#             bilans_par_date[str(date_c)] = {
#                 'siren': b.get('siren'),
#                 'date_cloture': date_c,
#                 'confidentialite': b.get('confidentiality', 'Public'),
#                 'liasses': contenu_liasse,
#                 'source': 'STRUCT'
#             }

#     for doc in pdfs + actes:
#         libelle = str(doc.get('libelle', '')).lower()
#         rdd_decisions = ""
#         if 'typeRdd' in doc:
#             rdd_decisions = " ".join([str(r.get('decision', '')).lower() for r in doc['typeRdd']])
#         date_c = doc.get('dateCloture') or doc.get('dateDepot')
#         if any(mot in libelle for mot in mots_cles) or any(mot in rdd_decisions for mot in mots_cles):
#             if date_c and str(date_c) not in bilans_par_date:
#                 bilans_par_date[str(date_c)] = {
#                     'siren': doc.get('siren'),
#                     'date_cloture': date_c,
#                     'confidentialite': doc.get('confidentiality', 'Public'),
#                     'liasses': doc,
#                     'source': 'PDF_DOCUMENT'
#                 }
#     return list(bilans_par_date.values())

# def refresh_inpi_token():
#     LOGIN_URL = "https://registre-national-entreprises.inpi.fr/api/sso/login"
#     payload = {"username": os.getenv("INPI_USERNAME"), "password": os.getenv("INPI_PASSWORD")}
#     try:
#         print("\n🔑 Tentative de connexion à l'INPI...")
#         response = requests.post(LOGIN_URL, json=payload, timeout=15)
#         response.raise_for_status()
#         token = response.json().get("token")
#         print(f"✅ Authentification réussie !")
#         return token
#     except Exception as e:
#         print(f"❌ Erreur d'authentification : {e}")
#         return None

# # --- 2. INITIALISATION ---
# load_dotenv()
# BACKUP_DIR = "backup_bilans_batch_2"
# INPUT_FILE = "siren_batch_20000_nouveaux.json"
# DATABASE_URL = os.getenv("NEON_DATABASE_URL")

# if not os.path.exists(BACKUP_DIR): os.makedirs(BACKUP_DIR)

# # Connexion BDD initiale
# def get_db_connection():
#     try:
#         c = psycopg2.connect(DATABASE_URL)
#         return c, c.cursor()
#     except Exception as e:
#         print(f"❌ Impossible de se connecter à la BDD : {e}")
#         return None, None

# conn, cur = get_db_connection()
# if not conn: exit()

# token = refresh_inpi_token()
# if not token: exit()
# HEADERS = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

# # --- 3. FILTRAGE RIGOUREUX ---
# with open(INPUT_FILE, 'r') as f:
#     tous_sirens_bruts = json.load(f)

# # On s'assure que tout est traité en STRING pour comparer pommes avec pommes
# deja_traites = {f.replace(".json", "") for f in os.listdir(BACKUP_DIR) if f.endswith(".json")}
# sirens_restants = [str(s) for s in tous_sirens_bruts if str(s) not in deja_traites]

# print("-" * 50)
# print(f"📊 REPRISE DU TRAVAIL :")
# print(f"✅ SIREN déjà écartés   : {len(deja_traites)}")
# print(f"🚀 SIREN à traiter      : {len(sirens_restants)}")
# print("-" * 50)

# # --- 4. BOUCLE D'ACQUISITION ---
# total_ajouts = 0
# idx = 0

# while idx < len(sirens_restants):
#     siren = sirens_restants[idx]
#     print(f"⏳ [{idx+1}/{len(sirens_restants)}] SIREN: {siren} | Ajouts: {total_ajouts}", end="\r")
    
#     url = f"https://registre-national-entreprises.inpi.fr/api/companies/{siren}/attachments"
    
#     try:
#         res = requests.get(url, headers=HEADERS, timeout=25)
        
#         # 4.1 GESTION AUTHENTIFICATION
#         if res.status_code == 401:
#             token = refresh_inpi_token()
#             if token:
#                 HEADERS["Authorization"] = f"Bearer {token}"
#                 continue 
#             else:
#                 time.sleep(60)
#                 continue

#         # 4.2 GESTION RATE LIMIT
#         if res.status_code == 429:
#             print(f"\n🚫 Rate Limit (429) sur {siren}. Pause 5min...")
#             time.sleep(300)
#             continue

#         # 4.3 GESTION SUCCÈS
#         if res.status_code == 200:
#             data = res.json()
#             bilans = extraire_bilans_dictionnaire(data)
            
#             if bilans:
#                 try:
#                     # Vérification connexion Neon avant chaque bloc d'insertion
#                     if conn.closed != 0:
#                         conn, cur = get_db_connection()

#                     for b in bilans:
#                         cur.execute("""
#                             INSERT INTO bilans_entreprises (siren, date_cloture, confidentialite, donnees_liasses)
#                             VALUES (%s, %s, %s, %s)
#                             ON CONFLICT (siren, date_cloture) DO NOTHING;
#                         """, (b['siren'], b['date_cloture'], b['confidentialite'], Json(b['liasses'])))
#                     conn.commit()
#                     total_ajouts += len(bilans)
#                 except Exception as e:
#                     print(f"\n⚠️ Erreur BDD sur {siren}: {e}")
#                     if conn: conn.rollback()
            
#             # Sauvegarde locale pour marquer comme "fait"
#             with open(f"{BACKUP_DIR}/{siren}.json", "w", encoding="utf-8") as f:
#                 json.dump(bilans, f, indent=2, ensure_ascii=False)
            
#             idx += 1

#         # 4.4 GESTION ERREURS CLASSIQUES
#         elif res.status_code == 404:
#             with open(f"{BACKUP_DIR}/{siren}.json", "w") as f: json.dump([], f)
#             idx += 1
#         else:
#             print(f"\n⚠️ Code {res.status_code} sur {siren}. Passage au suivant.")
#             idx += 1

#     except Exception as e:
#         print(f"\n❌ Erreur réseau/logique sur {siren}: {e}")
#         time.sleep(10)

#     # Respect du délai API (ajustable si tu es seul sur la clé)
#     time.sleep(5.1)

# if conn:
#     cur.close()
#     conn.close()
# print(f"\n\n✨ TERMINÉ ! {total_ajouts} nouveaux exercices ajoutés en base.")

---

#### Nouvelle boucle d'acquition pour 30 000 nouveaux SIREN

In [ ]:
# import os
# import json
# import time
# import requests
# import psycopg2
# from psycopg2.extras import Json
# from dotenv import load_dotenv

# # --- 1. FONCTIONS DE BASE ---

# def extraire_bilans_dictionnaire(data):
#     res_list = []
#     if not data: return res_list

#     saisis = data.get('bilansSaisis', [])
#     pdfs = data.get('bilans', [])
#     actes = data.get('actes', [])

#     bilans_par_date = {}
#     mots_cles = ["bilan", "comptes annuels", "liasse", "comptes de l'exercice", "clôture"]

#     for b in saisis:
#         date_c = b.get('dateCloture')
#         if date_c:
#             contenu_liasse = b.get('bilanSaisi', b)
#             bilans_par_date[str(date_c)] = {
#                 'siren': b.get('siren'),
#                 'date_cloture': date_c,
#                 'confidentialite': b.get('confidentiality', 'Public'),
#                 'liasses': contenu_liasse,
#                 'source': 'STRUCT'
#             }

#     for doc in pdfs + actes:
#         libelle = str(doc.get('libelle', '')).lower()
#         rdd_decisions = ""
#         if 'typeRdd' in doc:
#             rdd_decisions = " ".join([str(r.get('decision', '')).lower() for r in doc['typeRdd']])
#         date_c = doc.get('dateCloture') or doc.get('dateDepot')
#         if any(mot in libelle for mot in mots_cles) or any(mot in rdd_decisions for mot in mots_cles):
#             if date_c and str(date_c) not in bilans_par_date:
#                 bilans_par_date[str(date_c)] = {
#                     'siren': doc.get('siren'),
#                     'date_cloture': date_c,
#                     'confidentialite': doc.get('confidentiality', 'Public'),
#                     'liasses': doc,
#                     'source': 'PDF_DOCUMENT'
#                 }
#     return list(bilans_par_date.values())

# def refresh_inpi_token():
#     LOGIN_URL = "https://registre-national-entreprises.inpi.fr/api/sso/login"
#     payload = {"username": os.getenv("INPI_USERNAME"), "password": os.getenv("INPI_PASSWORD")}
#     try:
#         print("\n🔑 Tentative de connexion à l'INPI...")
#         response = requests.post(LOGIN_URL, json=payload, timeout=15)
#         response.raise_for_status()
#         token = response.json().get("token")
#         print(f"✅ Authentification réussie !")
#         return token
#     except Exception as e:
#         print(f"❌ Erreur d'authentification : {e}")
#         return None

# # --- 2. INITIALISATION ---
# load_dotenv()

# # CONFIGURATION DU BATCH
# INPUT_FILE = "siren_batch_30000_supplementaires.json" # Ton nouveau fichier
# BACKUP_DIR = "backup_bilans_batch_3"                  # Nouveau dossier pour ce batch
# DATABASE_URL = os.getenv("NEON_DATABASE_URL")

# if not os.path.exists(BACKUP_DIR): os.makedirs(BACKUP_DIR)

# def get_db_connection():
#     try:
#         c = psycopg2.connect(DATABASE_URL)
#         return c, c.cursor()
#     except Exception as e:
#         print(f"❌ Erreur connexion BDD : {e}")
#         return None, None

# conn, cur = get_db_connection()
# if not conn: exit()

# token = refresh_inpi_token()
# if not token: exit()
# HEADERS = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

# # --- 3. FILTRAGE RIGOUREUX ---
# with open(INPUT_FILE, 'r') as f:
#     tous_sirens_bruts = json.load(f)

# # On filtre par rapport à ce qui est déjà dans le dossier de backup du batch 3
# deja_traites = {f.replace(".json", "") for f in os.listdir(BACKUP_DIR) if f.endswith(".json")}
# sirens_restants = [str(s) for s in tous_sirens_bruts if str(s) not in deja_traites]

# print("-" * 50)
# print(f"📊 LANCEMENT BATCH #3 (30k) :")
# print(f"✅ SIREN déjà faits dans ce dossier : {len(deja_traites)}")
# print(f"🚀 SIREN restants à traiter        : {len(sirens_restants)}")
# print("-" * 50)

# # --- 4. BOUCLE D'ACQUISITION ---
# total_ajouts = 0
# idx = 0

# while idx < len(sirens_restants):
#     siren = sirens_restants[idx]
#     print(f"⏳ [{idx+1}/{len(sirens_restants)}] SIREN: {siren} | Total Ajouts: {total_ajouts}", end="\r")
    
#     url = f"https://registre-national-entreprises.inpi.fr/api/companies/{siren}/attachments"
    
#     try:
#         res = requests.get(url, headers=HEADERS, timeout=25)
        
#         # 4.1 GESTION AUTHENTIFICATION
#         if res.status_code == 401:
#             token = refresh_inpi_token()
#             if token:
#                 HEADERS["Authorization"] = f"Bearer {token}"
#                 continue 
#             else:
#                 time.sleep(60); continue

#         # 4.2 GESTION RATE LIMIT
#         if res.status_code == 429:
#             print(f"\n🚫 Rate Limit (429). Pause 5min...")
#             time.sleep(300); continue

#         # 4.3 GESTION SUCCÈS
#         if res.status_code == 200:
#             data = res.json()
#             bilans = extraire_bilans_dictionnaire(data)
            
#             if bilans:
#                 try:
#                     # RECONNEXION AGRESSIVE SI BESOIN (Correction de ton erreur SSL précédente)
#                     try:
#                         cur.execute("SELECT 1")
#                     except (psycopg2.OperationalError, psycopg2.InterfaceError, psycopg2.DatabaseError):
#                         print("\n🔄 Reconnexion à Neon suite à coupure SSL...")
#                         conn, cur = get_db_connection()

#                     for b in bilans:
#                         cur.execute("""
#                             INSERT INTO bilans_entreprises (siren, date_cloture, confidentialite, donnees_liasses)
#                             VALUES (%s, %s, %s, %s)
#                             ON CONFLICT (siren, date_cloture) DO NOTHING;
#                         """, (b['siren'], b['date_cloture'], b['confidentialite'], Json(b['liasses'])))
#                     conn.commit()
#                     total_ajouts += len(bilans)
#                 except Exception as e:
#                     print(f"\n⚠️ Erreur insertion BDD {siren}: {e}")
#                     if conn: conn.rollback()
            
#             with open(f"{BACKUP_DIR}/{siren}.json", "w", encoding="utf-8") as f:
#                 json.dump(bilans, f, indent=2, ensure_ascii=False)
            
#             idx += 1

#         elif res.status_code == 404:
#             with open(f"{BACKUP_DIR}/{siren}.json", "w") as f: json.dump([], f)
#             idx += 1
#         else:
#             print(f"\n⚠️ Code {res.status_code} sur {siren}. Suivant.")
#             idx += 1

#     except Exception as e:
#         print(f"\n❌ Erreur réseau/logique sur {siren}: {e}")
#         time.sleep(10)

#     time.sleep(5.1)

# if conn:
#     cur.close(); conn.close()
# print(f"\n\n✨ BATCH #3 TERMINÉ ! {total_ajouts} nouveaux exercices en base.")

---

#### Ajout de 100 000 nouveaux SIREN

In [ ]:
# import os
# import json
# import time
# import requests
# import psycopg2
# from psycopg2.extras import Json
# from dotenv import load_dotenv

# # --- 1. FONCTIONS DE BASE (Inchangées) ---

# def extraire_bilans_dictionnaire(data):
#     res_list = []
#     if not data: return res_list
#     saisis = data.get('bilansSaisis', [])
#     pdfs = data.get('bilans', [])
#     actes = data.get('actes', [])
#     bilans_par_date = {}
#     mots_cles = ["bilan", "comptes annuels", "liasse", "comptes de l'exercice", "clôture"]

#     for b in saisis:
#         date_c = b.get('dateCloture')
#         if date_c:
#             contenu_liasse = b.get('bilanSaisi', b)
#             bilans_par_date[str(date_c)] = {
#                 'siren': b.get('siren'),
#                 'date_cloture': date_c,
#                 'confidentialite': b.get('confidentiality', 'Public'),
#                 'liasses': contenu_liasse,
#                 'source': 'STRUCT'
#             }
#     for doc in pdfs + actes:
#         libelle = str(doc.get('libelle', '')).lower()
#         rdd_decisions = ""
#         if 'typeRdd' in doc:
#             rdd_decisions = " ".join([str(r.get('decision', '')).lower() for r in doc['typeRdd']])
#         date_c = doc.get('dateCloture') or doc.get('dateDepot')
#         if any(mot in libelle for mot in mots_cles) or any(mot in rdd_decisions for mot in mots_cles):
#             if date_c and str(date_c) not in bilans_par_date:
#                 bilans_par_date[str(date_c)] = {
#                     'siren': doc.get('siren'),
#                     'date_cloture': date_c,
#                     'confidentialite': doc.get('confidentiality', 'Public'),
#                     'liasses': doc,
#                     'source': 'PDF_DOCUMENT'
#                 }
#     return list(bilans_par_date.values())

# def refresh_inpi_token():
#     LOGIN_URL = "https://registre-national-entreprises.inpi.fr/api/sso/login"
#     payload = {"username": os.getenv("INPI_USERNAME"), "password": os.getenv("INPI_PASSWORD")}
#     try:
#         print("\n🔑 Tentative de connexion à l'INPI...")
#         response = requests.post(LOGIN_URL, json=payload, timeout=20)
#         response.raise_for_status()
#         token = response.json().get("token")
#         print(f"✅ Authentification réussie !")
#         return token
#     except Exception as e:
#         print(f"❌ Erreur d'authentification : {e}")
#         return None

# # --- 2. INITIALISATION DU BATCH #4 ---
# load_dotenv()

# # CHANGEMENT ICI
# INPUT_FILE = "siren_batch_100000_supplementaires.json" 
# BACKUP_DIR = "backup_bilans_batch_4"                  
# DATABASE_URL = os.getenv("NEON_DATABASE_URL")

# if not os.path.exists(BACKUP_DIR): os.makedirs(BACKUP_DIR)

# def get_db_connection():
#     try:
#         c = psycopg2.connect(DATABASE_URL)
#         return c, c.cursor()
#     except Exception as e:
#         print(f"❌ Erreur connexion BDD : {e}")
#         return None, None

# conn, cur = get_db_connection()
# if not conn: exit()

# token = refresh_inpi_token()
# if not token: exit()
# HEADERS = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

# # --- 3. FILTRAGE RIGOUREUX ---
# with open(INPUT_FILE, 'r') as f:
#     tous_sirens_bruts = json.load(f)

# print("🔍 Analyse du dossier de backup (cela peut prendre du temps pour 100k)...")
# deja_traites = {f.replace(".json", "") for f in os.listdir(BACKUP_DIR) if f.endswith(".json")}
# sirens_restants = [str(s) for s in tous_sirens_bruts if str(s) not in deja_traites]

# print("-" * 50)
# print(f"🚀 LANCEMENT DU GROS BATCH #4 (100k) :")
# print(f"✅ SIREN déjà écartés   : {len(deja_traites)}")
# print(f"🚀 SIREN à traiter      : {len(sirens_restants)}")
# print("-" * 50)

# # --- 4. BOUCLE D'ACQUISITION ---
# total_ajouts = 0
# idx = 0

# while idx < len(sirens_restants):
#     siren = sirens_restants[idx]
#     # Affichage plus compact pour les logs
#     print(f"⏳ [{idx+1}/{len(sirens_restants)}] SIREN: {siren} | Ajouts: {total_ajouts}", end="\r")
    
#     url = f"https://registre-national-entreprises.inpi.fr/api/companies/{siren}/attachments"
    
#     try:
#         res = requests.get(url, headers=HEADERS, timeout=30)
        
#         # 4.1 GESTION AUTHENTIFICATION (Token expiré)
#         if res.status_code == 401:
#             token = refresh_inpi_token()
#             if token:
#                 HEADERS["Authorization"] = f"Bearer {token}"
#                 continue # Retente le SIREN actuel
#             else:
#                 print("\n😴 Échec Refresh Token. Pause 5min...")
#                 time.sleep(300); continue

#         # 4.2 GESTION RATE LIMIT
#         if res.status_code == 429:
#             print(f"\n🚫 Rate Limit (429). Pause de 5min pour souffler...")
#             time.sleep(300); continue

#         # 4.3 GESTION SUCCÈS
#         if res.status_code == 200:
#             data = res.json()
#             bilans = extraire_bilans_dictionnaire(data)
            
#             if bilans:
#                 try:
#                     # Test de connexion Neon
#                     try:
#                         cur.execute("SELECT 1")
#                     except (psycopg2.OperationalError, psycopg2.InterfaceError, psycopg2.DatabaseError):
#                         print("\n🔄 Reconnexion à Neon...")
#                         conn, cur = get_db_connection()

#                     for b in bilans:
#                         cur.execute("""
#                             INSERT INTO bilans_entreprises (siren, date_cloture, confidentialite, donnees_liasses)
#                             VALUES (%s, %s, %s, %s)
#                             ON CONFLICT (siren, date_cloture) DO NOTHING;
#                         """, (b['siren'], b['date_cloture'], b['confidentialite'], Json(b['liasses'])))
#                     conn.commit()
#                     total_ajouts += len(bilans)
#                 except Exception as e:
#                     print(f"\n⚠️ Erreur BDD sur {siren}: {e}")
#                     if conn: conn.rollback()
            
#             # Marquage local (crucial pour la reprise)
#             with open(f"{BACKUP_DIR}/{siren}.json", "w", encoding="utf-8") as f:
#                 json.dump(bilans, f, indent=2, ensure_ascii=False)
            
#             idx += 1

#         elif res.status_code == 404:
#             with open(f"{BACKUP_DIR}/{siren}.json", "w") as f: json.dump([], f)
#             idx += 1
#         else:
#             print(f"\n⚠️ Code {res.status_code} sur {siren}. Pause 10s...")
#             time.sleep(10)
#             idx += 1

#     except Exception as e:
#         print(f"\n❌ Erreur réseau sur {siren}: {e}. Pause 15s...")
#         time.sleep(15)

#     # Délai de courtoisie INPI
#     time.sleep(6.0)

# if conn:
#     cur.close(); conn.close()
# print(f"\n\n✨ BATCH #4 TERMINÉ ! {total_ajouts} nouveaux bilans en base.")

In [ ]:
import os
import json
import time
import random
import requests
import psycopg2
from psycopg2.extras import Json
from dotenv import load_dotenv

# --- 1. FONCTIONS DE BASE ---

def extraire_bilans_dictionnaire(data):
    res_list = []
    if not data: return res_list
    saisis = data.get('bilansSaisis', [])
    pdfs = data.get('bilans', [])
    actes = data.get('actes', [])
    bilans_par_date = {}
    mots_cles = ["bilan", "comptes annuels", "liasse", "comptes de l'exercice", "clôture"]

    for b in saisis:
        date_c = b.get('dateCloture')
        if date_c:
            contenu_liasse = b.get('bilanSaisi', b)
            bilans_par_date[str(date_c)] = {
                'siren': b.get('siren'),
                'date_cloture': date_c,
                'confidentialite': b.get('confidentiality', 'Public'),
                'liasses': contenu_liasse,
                'source': 'STRUCT'
            }
    for doc in pdfs + actes:
        libelle = str(doc.get('libelle', '')).lower()
        rdd_decisions = ""
        if 'typeRdd' in doc:
            rdd_decisions = " ".join([str(r.get('decision', '')).lower() for r in doc['typeRdd']])
        date_c = doc.get('dateCloture') or doc.get('dateDepot')
        if any(mot in libelle for mot in mots_cles) or any(mot in rdd_decisions for mot in mots_cles):
            if date_c and str(date_c) not in bilans_par_date:
                bilans_par_date[str(date_c)] = {
                    'siren': doc.get('siren'),
                    'date_cloture': date_c,
                    'confidentialite': doc.get('confidentiality', 'Public'),
                    'liasses': doc,
                    'source': 'PDF_DOCUMENT'
                }
    return list(bilans_par_date.values())

def refresh_inpi_token():
    LOGIN_URL = "https://registre-national-entreprises.inpi.fr/api/sso/login"
    payload = {"username": os.getenv("INPI_USERNAME"), "password": os.getenv("INPI_PASSWORD")}
    try:
        print("\n🔑 Tentative de connexion à l'INPI...")
        response = requests.post(LOGIN_URL, json=payload, timeout=20)
        response.raise_for_status()
        token = response.json().get("token")
        print(f"✅ Authentification réussie !")
        return token
    except Exception as e:
        print(f"❌ Erreur d'authentification : {e}")
        return None

# --- 2. INITIALISATION ---
load_dotenv()
INPUT_FILE = "siren_batch_100000_supplementaires.json" 
BACKUP_DIR = "backup_bilans_batch_4"                  
DATABASE_URL = os.getenv("NEON_DATABASE_URL")

if not os.path.exists(BACKUP_DIR): os.makedirs(BACKUP_DIR)

def get_db_connection():
    try:
        c = psycopg2.connect(DATABASE_URL)
        return c, c.cursor()
    except Exception as e:
        print(f"❌ Erreur BDD : {e}"); return None, None

conn, cur = get_db_connection()
token = refresh_inpi_token()

if not token or not conn: 
    print("Abort: Token ou Connexion BDD manquante.")
    exit()

HEADERS = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

# --- 3. FILTRAGE ---
with open(INPUT_FILE, 'r') as f:
    tous_sirens_bruts = json.load(f)

deja_traites = {f.replace(".json", "") for f in os.listdir(BACKUP_DIR) if f.endswith(".json")}
sirens_restants = [str(s) for s in tous_sirens_bruts if str(s) not in deja_traites]

print("-" * 50)
print(f"🚀 REPRISE BATCH #4 | Restants : {len(sirens_restants)}")
print("-" * 50)

# --- 4. BOUCLE D'ACQUISITION ---
total_ajouts = 0
idx = 0
compteur_429 = 0
MAX_429 = 5 

while idx < len(sirens_restants):
    siren = sirens_restants[idx]
    print(f"⏳ [{idx+1}/{len(sirens_restants)}] SIREN: {siren} | Ajouts: {total_ajouts}", end="\r")
    
    # --- AJOUT : La "Pause Café" toutes les 100 requêtes ---
    if idx > 0 and idx % 100 == 0:
        pause_longue = random.randint(180, 300)
        print(f"\n☕ Pause de sécurité (tous les 100) : {pause_longue}s...")
        time.sleep(pause_longue)

    url = f"https://registre-national-entreprises.inpi.fr/api/companies/{siren}/attachments"
    
    try:
        # Connection: close est parfait, on garde
        res = requests.get(url, headers={**HEADERS, "Connection": "close"}, timeout=30)
        
        # GESTION RATE LIMIT
        if res.status_code == 429:
            compteur_429 += 1
            print(f"\n🚫 Rate Limit ({compteur_429}/{MAX_429}). Pause 10 min...")
            if compteur_429 >= MAX_429:
                print("\n🚨 ARRÊT : Trop d'erreurs 429. IP bloquée.")
                break
            time.sleep(600)
            continue

        # GESTION AUTHENTIFICATION
        if res.status_code == 401:
            token = refresh_inpi_token()
            if token:
                HEADERS["Authorization"] = f"Bearer {token}"
                continue
            else:
                print("\n❌ Impossible de refresh le token. Arrêt.")
                break

        # GESTION SUCCÈS (200) OU NOT FOUND (404)
        if res.status_code in [200, 404]:
            compteur_429 = 0 # On remet à zéro car l'IP répond bien
            
            if res.status_code == 200:
                data = res.json()
                bilans = extraire_bilans_dictionnaire(data)
                
                if bilans:
                    try:
                        # On vérifie la connexion BDD à chaque insertion
                        try:
                            cur.execute("SELECT 1")
                        except:
                            conn, cur = get_db_connection()

                        for b in bilans:
                            cur.execute("""
                                INSERT INTO bilans_entreprises (siren, date_cloture, confidentialite, donnees_liasses)
                                VALUES (%s, %s, %s, %s)
                                ON CONFLICT (siren, date_cloture) DO NOTHING;
                            """, (b['siren'], b['date_cloture'], b['confidentialite'], Json(b['liasses'])))
                        conn.commit()
                        total_ajouts += len(bilans)
                    except Exception as e:
                        print(f"\n⚠️ Erreur BDD sur {siren}: {e}")
                        if conn: conn.rollback()
                
                with open(f"{BACKUP_DIR}/{siren}.json", "w", encoding="utf-8") as f:
                    json.dump(bilans, f, indent=2, ensure_ascii=False)
            
            else: # Cas 404
                with open(f"{BACKUP_DIR}/{siren}.json", "w") as f: json.dump([], f)
            
            idx += 1

            time.sleep(8.0 + random.uniform(0.5, 4.5))

        else:
            print(f"\n⚠️ Code {res.status_code} sur {siren}. Pause 30s...")
            time.sleep(30)
            idx += 1

    except Exception as e:
        print(f"\n❌ Erreur réseau sur {siren}: {e}. Pause 20s...")
        time.sleep(20)

---

#### Test de SIREN en cas de crash

In [ ]:
# test_siren = "389091893" # Un de tes SAS/SARL filtrés
# url = f"https://registre-national-entreprises.inpi.fr/api/companies/{test_siren}/attachments"
# res = requests.get(url, headers=HEADERS)
# data = res.json()

# print(f"Structure pour {test_siren}:")
# print(f"Clefs présentes : {data.keys()}")
# if 'bilansSaisis' in data:
#     print(f"Nombre de bilans saisis : {len(data['bilansSaisis'])}")
#     if len(data['bilansSaisis']) > 0:
#         print(f"Clefs du premier bilan : {data['bilansSaisis'][0].keys()}")

---

##### Test de connection suite problèmes

In [ ]:
# test_siren = "511287146"
# url = f"https://registre-national-entreprises.inpi.fr/api/companies/{test_siren}/attachments"
# res = requests.get(url, headers=HEADERS)
# data = res.json()

# # On regarde ce que l'extracteur voit vraiment
# resultat_test = extraire_bilans_dictionnaire(data)
# print(f"DEBUG pour {test_siren} :")
# print(f"Nombre d'exercices trouvés : {len(resultat_test)}")
# if len(resultat_test) > 0:
#     print(f"Premier exercice : {resultat_test[0].keys()}")
#     # Est-ce que les liasses contiennent des chiffres ?
#     print(f"Contenu 'liasses' : {list(resultat_test[0]['liasses'].keys())[:5]}")

In [ ]:

# siren_test = "775670417" 

# # 2. Utilise ton token actuel (assure-toi qu'il est encore valide)
# headers = {"Authorization": f"Bearer {token}"} 
# url = f"https://registre-national-entreprises.inpi.fr/api/companies/{siren_test}/attachments"

# print(f"🔍 Test de l'API pour le SIREN : {siren_test}")
# response = requests.get(url, headers=headers)

# print(f"Status Code : {response.status_code}")

# if response.status_code == 200:
#     data = response.json()
#     print("--- RÉPONSE BRUTE ---")
#     print(data) # On veut voir s'il y a des clés comme 'attachments', 'documents', etc.
    
#     if isinstance(data, list):
#         print(f"✅ Reçu une liste de {len(data)} éléments.")
#     elif isinstance(data, dict):
#         print(f"✅ Reçu un dictionnaire. Clés présentes : {data.keys()}")
#     else:
#         print("⚠️ Format de réponse inconnu.")
# else:
#     print(f"❌ Erreur API : {response.text}")

---

#### Backup des bilans dans Néon suite crash de l'envoi

In [ ]:
# # --- 1. CONFIGURATION ---
# load_dotenv()
# BACKUP_DIR = "backup_bilans"
# DATABASE_URL = os.getenv("NEON_DATABASE_URL")

# def sync_backups_to_neon():
#     try:
#         # Connexion à Neon
#         conn = psycopg2.connect(DATABASE_URL)
#         cur = conn.cursor()
#         print("✅ Connecté à Neon. Début de la synchronisation...")

#         # Lister tous les fichiers JSON du dossier backup
#         fichiers = [f for f in os.listdir(BACKUP_DIR) if f.endswith('.json')]
#         total = len(fichiers)
#         success_count = 0
#         skip_count = 0

#         for i, nom_fichier in enumerate(fichiers):
#             siren = nom_fichier.replace(".json", "")
#             print(f"🔄 [{i+1}/{total}] Synchronisation SIREN: {siren}", end="\r")

#             with open(os.path.join(BACKUP_DIR, nom_fichier), 'r', encoding='utf-8') as f:
#                 bilans = json.load(f)

#             if bilans:
#                 for b in bilans:
#                     try:
#                         cur.execute("""
#                             INSERT INTO bilans_entreprises (siren, date_cloture, confidentialite, donnees_liasses)
#                             VALUES (%s, %s, %s, %s)
#                             ON CONFLICT (siren, date_cloture) DO NOTHING;
#                         """, (b['siren'], b['date_cloture'], b['confidentialite'], Json(b['liasses'])))
#                         success_count += 1
#                     except Exception as e:
#                         print(f"\n❌ Erreur SQL sur {siren}: {e}")
#                         conn.rollback()
#                 conn.commit()
#             else:
#                 skip_count += 1

#         print(f"\n\n✨ SYNCHRO TERMINÉE !")
#         print(f"📊 Bilans insérés (ou déjà présents) : {success_count}")
#         print(f"ℹ️ SIREN sans bilans ignorés : {skip_count}")

#         cur.close()
#         conn.close()

#     except Exception as e:
#         print(f"❌ Erreur de connexion ou de lecture : {e}")

# if __name__ == "__main__":
#     sync_backups_to_neon()